## RAG Day 3

### Expert Question Answerer for InsureLLM

LangChain 1.0 implementation of a RAG pipeline.

Using the VectorStore we created last time (with HuggingFace `all-MiniLM-L6-v2`)

In [8]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
import os
from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_ollama import ChatOllama
import gradio as gr

In [5]:
DB_NAME = "vector_db"

load_dotenv(override=True)
ollama_url = os.getenv('OLLAMA_BASE_URL')
# print(ollama_url)

MODEL = "gpt-oss:20b"

### Connect to Chroma; use Hugging Face all-MiniLM-L6-v2

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)

### Set up the 2 key LangChain objects: retriever and llm

#### A sidebar on "temperature":
- Controls how diverse the output is
- A temperature of 0 means that the output should be predictable
- Higher temperature for more variety in answers

Some people describe temperature as being like 'creativity' but that's not quite right
- It actually controls which tokens get selected during inference
- temperature=0 means: always select the token with highest probability
- temperature=1 usually means: a token with 10% probability should be picked 10% of the time

Note: a temperature of 0 doesn't mean outputs will always be reproducible. You also need to set a random seed. We will do that in weeks 6-8. (Even then, it's not always reproducible.)

Note 2: if you want creativity, use the System Prompt!

In [9]:
retriever = vectorstore.as_retriever()

llm = ChatOllama(
    model=MODEL,
    temperature=0,
)

### These LangChain objects implement the method `invoke()`

In [10]:
retriever.invoke("Who is Avery?")

[Document(id='47cd692e-f745-48d2-b8ff-386f1fa11362', metadata={'doc_type': 'employees', 'source': 'knowledge-base/employees/Avery Lancaster.md'}, page_content="## Other HR Notes\n- **Professional Development**: Avery has actively participated in leadership training programs and industry conferences, representing Insurellm and fostering partnerships.  \n- **Diversity & Inclusion Initiatives**: Avery has championed a commitment to diversity in hiring practices, seeing visible improvements in team representation since 2021.  \n- **Work-Life Balance**: Feedback revealed concerns regarding work-life balance, which Avery has approached by implementing flexible working conditions and ensuring regular check-ins with the team.\n- **Community Engagement**: Avery led community outreach efforts, focusing on financial literacy programs, particularly aimed at underserved populations, improving Insurellm's corporate social responsibility image.  \n\nAvery Lancaster has demonstrated resilience and ada

In [11]:
llm.invoke("Who is Avery?")

AIMessage(content='I’m not sure which Avery you’re referring to—there are many people, characters, and even places named Avery. Could you let me know a bit more about the context? For example:\n\n- Is Avery a real person (e.g., a public figure, athlete, scientist)?\n- Is Avery a fictional character (e.g., from a book, movie, TV show)?\n- Are you asking about a place or thing named Avery (e.g., a town, a company)?\n\nAny extra details you can share will help me give you the most accurate answer.', additional_kwargs={}, response_metadata={'model': 'gpt-oss:20b', 'created_at': '2026-02-05T11:50:44.469928111Z', 'done': True, 'done_reason': 'stop', 'total_duration': 3348524657, 'load_duration': 243931097, 'prompt_eval_count': 71, 'prompt_eval_duration': 112519332, 'eval_count': 220, 'eval_duration': 2777429954, 'model_name': 'gpt-oss:20b', 'model_provider': 'ollama'}, id='lc_run--955412c0-d777-41f0-ba54-7c898b57e28c-0', usage_metadata={'input_tokens': 71, 'output_tokens': 220, 'total_tokens

## Time to put this together!

In [12]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [13]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [14]:
answer_question("Who is Averi Lancaster?", [])

'**Avery Lancaster** (sometimes misspelled as “Averi”) is the Co‑Founder and Chief Executive Officer (CEO) of Insurellm.\n\n| Detail | Information |\n|--------|-------------|\n| **Date of Birth** | March\u202f15,\u202f1985 |\n| **Location** | San\u202fFrancisco, California |\n| **Current Salary** | $225,000 |\n| **Role at Insurellm** | Co‑Founder & CEO (2015‑present) |\n| **Career Highlights** | • Launched Insurellm in 2015 and steered it to become a leading InsurTech provider.<br>• Prior to Insurellm, served as Senior Product Manager at Innovate Insurance Solutions (2013‑2015), creating tech‑focused insurance products.<br>• In January\u202f2021, took on the role of Senior Data Engineer, leading a project that cut data retrieval times by 30% and mentoring junior engineers. (This dual role underscores her hands‑on leadership style.) |\n| **Leadership Style** | Known for innovative strategies, strong risk‑management acumen, and a knack for bridging technical teams with business objective

## What could possibly come next? 😂

In [15]:
gr.ChatInterface(answer_question).launch()

/home/tanishq/Code/llm_engineering/.venv/lib/python3.12/site-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## Admit it - you thought RAG would be more complicated than that!!